# Part 0 - Introduction to Julia and Catalyst

This notebook gives a quick Julia + Catalyst onboarding before the main workshop parts.


## Package and environment management in Julia

In this workshop, all notebooks share one local Julia environment (the `Project.toml` and `Manifest.toml` in the top folder).

If this is your first time running the workshop, activate the environment and instantiate it:


In [ ]:
using Pkg
Pkg.activate(".")
Pkg.instantiate()


We now load the packages used in this notebook.


In [ ]:
using Catalyst, OrdinaryDiffEq, Plots


# Part 1.1 - Basic Modelling

We create a simple birth/death model where a species `X` is produced at rate `p` and degraded at rate `d`.


In [ ]:
crn = @reaction_network begin
    p, 0 --> X
    d, X --> 0
end


To simulate, we provide:
- initial conditions,
- a time span,
- and parameter values.


In [ ]:
u0 = [:X => 2.0]
tspan = (0.0, 10.0)
ps = [:p => 2.0, :d => 0.2]

ode_prob = ODEProblem(crn, u0, tspan, ps)
ode_sol = solve(ode_prob, Tsit5())


In [ ]:
plot(ode_sol; xlabel = "t", ylabel = "concentration", title = "ODE simulation")


# Part 1.2 - Basic Simulations

Catalyst models can be simulated as ODEs, SDEs, and jump (SSA/Gillespie-type) systems.


## SDE simulation

For SDEs, we create an `SDEProblem` and choose an SDE solver.


In [ ]:
using StochasticDiffEq

sde_prob = SDEProblem(crn, u0, tspan, ps)
sde_sol = solve(sde_prob, ImplicitEM(); dt = 0.01)
plot(sde_sol; xlabel = "t", ylabel = "concentration", title = "SDE simulation")


## Jump simulation

For jump simulations in Catalyst v16, we can directly build a jump problem from
`JumpProblem(rn, u0, tspan, ps)`.

Note that molecule counts should generally be integers for SSA models.


In [ ]:
using JumpProcesses

u0_jump = [:X => 2]
jump_prob = JumpProblem(crn, u0_jump, tspan, ps)
jump_sol = solve(jump_prob, SSAStepper())

plot(jump_sol; xlabel = "t", ylabel = "molecule count", title = "Jump simulation")
